## Customer Data Integration Pipeline 
### From Sports Bar (Chilled Company) -> Atlikon (Parant Company) At Gold Layer

A Medallion Architecture ETL pipeline built with PySpark and Delta Lake to integrate customer data from an acquired subsidiary into a centralized enterprise data warehouse. The pipeline handles data ingestion, cleansing, standardization, and dimensional merging across Bronze, Silver, and Gold layers.

### WARNING:

The Gold layer merge process uses an unconditional update on match. If a matching customer code exists in the target dimension table, all existing customer attributes in the enterprise database will be overwritten by the incoming source data. Furthermore, missing city values are imputed using static ID mapping and fallback defaults to prevent string formatting errors during transformation.

In [0]:
# ==========================================
# 1. Module Imports & Dependencies
# ==========================================
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col
from delta.tables import DeltaTable

In [0]:

# ==========================================
# 2. Dynamic Notebook Parameters (Widgets)
# ==========================================
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_Sourse', 'customers', 'Data Source')

# Extract parameter values
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_Sourse')

In [0]:
%run "../Utitlites"

In [0]:
# ==========================================
# 3. Read Raw CSV Data & Append File Metadata
# ==========================================
path_file = chilled_customers_path_file

df_raw = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .option('sep', ',')
    .load(f'path_file'+'customers.csv')
    .select(
        '*', 
        '_metadata.file_name', 
        '_metadata.file_size'
    )
    .withColumn('read_timestamp', F.current_timestamp())
)

## Bronze Processing

In [0]:
# ==========================================
# 4. Save Raw Data to Bronze Layer (Delta Format)
# ==========================================
df_raw.write \
    .format('delta') \
    .option('delta.enableChangeDataFeed', True) \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.bronze.{data_source}')

# Load Bronze table for cleansing transformations
df_bronze = spark.sql(f'SELECT * FROM {catalog}.bronze.{data_source}')
display(df_bronze.limit(10))

customer_id,customer_name,city,file_name,file_size,read_timestamp
789201,FitFuel Market,Bengaluru,customers.csv,1404,2026-09-25T12:06:36.552Z
789202,FitFuel Market,Hyderabad,customers.csv,1404,2026-09-25T12:06:36.552Z
789203,FitFuel Market,New Delhi,customers.csv,1404,2026-09-25T12:06:36.552Z
789301,Athlete's Choice Store,Bengaluru,customers.csv,1404,2026-09-25T12:06:36.552Z
789303,Athlete's Choice Store,New Delhi,customers.csv,1404,2026-09-25T12:06:36.552Z
789101,Endurance Foods,Bengalore,customers.csv,1404,2026-09-25T12:06:36.552Z
789102,Endurance Foods,Hyderabad,customers.csv,1404,2026-09-25T12:06:36.552Z
789103,Endurance Foods,New Delhi,customers.csv,1404,2026-09-25T12:06:36.552Z
789121,HydroBoost Nutrition,Hyderabad,customers.csv,1404,2026-09-25T12:06:36.552Z
789122,HydroBoost Nutrition,New Delhi,customers.csv,1404,2026-09-25T12:06:36.552Z


In [0]:
# ==========================================
# 5. Data Cleansing & Standardization Operations
# ==========================================

# A. Deduplicate records based on primary customer identifier
df_bronze = df_bronze.dropDuplicates(['customer_id'])

# B. Trim whitespace and apply Proper Case (InitCap) to customer names
df_bronze = df_bronze.withColumn('customer_name', F.trim(F.col('customer_name')))
df_bronze = df_bronze.withColumn('customer_name', F.initcap(F.col('customer_name')))

# C. City Name Standardization Mapping
city_clean_map = {
    "bangalore": "Bengaluru", "banglore": "Bengaluru", "bangaluru": "Bengaluru",
    "bengaluru": "Bengaluru", "bengaluru city": "Bengaluru", "blru": "Bengaluru",
    "blr": "Bengaluru", "bngl": "Bengaluru",
    "delhi": "New Delhi", "newdelhi": "New Delhi", "new delhi": "New Delhi",
    "n. delhi": "New Delhi", "ndelhi": "New Delhi", "dlh": "New Delhi",
    "del": "New Delhi", "dilli": "New Delhi",
    "hyderabad": "Hyderabad", "hydrabad": "Hyderabad", "hyderbad": "Hyderabad",
    "hyderabadd": "Hyderabad", "hyd": "Hyderabad", "hyderabad city": "Hyderabad",
    "secunderabad": "Hyderabad"
}

df_bronze = (
    df_bronze.replace(city_clean_map, subset=['city'])
    .withColumn(
        'city',
        F.when(F.col('city').isNull(), None)
         .when(F.col('city').isin(list(city_clean_map.values())), F.col('city'))
         .otherwise(None)
    )
)

# D. Manual City Imputation for Specific Customer IDs
customer_city_map = {
    789101: "Bengaluru", 789403: "Bengaluru", 789503: "Bengaluru",
    789221: "New Delhi", 789520: "New Delhi", 789521: "New Delhi", 789522: "New Delhi",
    789420: "Hyderabad", 789421: "Hyderabad", 789422: "Hyderabad", 789603: "Hyderabad"
}

fixed_city_df = spark.createDataFrame(
    [(k, v) for k, v in customer_city_map.items()],
    ['customer_id', 'new_city']
)

df_bronze = (
    df_bronze
    .join(fixed_city_df, 'customer_id', 'left')
    .withColumn('city', F.coalesce(F.col('city'), F.col('new_city')))
    .drop('new_city')
)

# E. Cast Customer ID, append City suffix to name, and inject Business Metadata
df_bronze = (
    df_bronze
    .withColumn('customer_id', F.col('customer_id').cast('string'))
    .withColumn('customer_name', F.concat_ws('-', F.col('customer_name'), F.coalesce(F.col('city'), F.lit('Unknown'))))
    .withColumn('market', F.lit('India'))
    .withColumn('platform', F.lit('Sports Bar'))
    .withColumn('channel', F.lit('Acquisition'))
)

## Silver Processing

In [0]:
# ==========================================
# 6. Save Cleansed Data to Silver Layer
# ==========================================
df_bronze.write \
    .format('delta') \
    .option('delta.enableChangeDataFeed', True) \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.silver.sb_{data_source}')


## Gold Processing

In [0]:
# ==========================================
# 7. Merge Silver Data into Gold Dimension Table
# ==========================================
p_customers = DeltaTable.forName(spark, f'{catalog}.gold.dim_customers')

ch_customers = spark.table(f'{catalog}.silver.sb_{data_source}').select(
    F.col('customer_id').alias('customer_code'),
    F.col('customer_name').alias('customer'),
    'market',
    'platform',
    'channel'
)

# Execute Atomic Delta Merge Operation
p_customers.alias('target').merge(
    source=ch_customers.alias('source'),
    condition='target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Data Quality & Validation Checks

In [0]:
# ==========================================
# 8. Data Quality & Validation Suite (Multi-Layer Checks)
# ==========================================

print("=== LAYER 1: BRONZE (Source Audit) ===")
df_bronze_raw = spark.sql(f'SELECT * FROM {catalog}.bronze.{data_source}')
duplicates_bronze = df_bronze_raw.groupBy("customer_id").count().filter("count > 1").count()
untrimmed_bronze = df_bronze_raw.filter(F.col('customer_name') != F.trim(F.col('customer_name'))).count()
print(f"Raw duplicate customer IDs in Bronze: {duplicates_bronze}")
print(f"Raw records with untrimmed whitespace: {untrimmed_bronze}")

print("\n=== LAYER 2: SILVER (Cleansing Validation) ===")
df_silver = spark.table(f"{catalog}.silver.sb_{data_source}")
null_cities = df_silver.filter(F.col('city').isNull()).count()
print(f"Remaining records with NULL city in Silver: {null_cities}")
print("Distinct Cities in Silver Layer:")
df_silver.select('city').distinct().show()

print("\n=== LAYER 3: GOLD (Dimensional Model Validation) ===")
df_gold = spark.table(f"{catalog}.gold.dim_customers")

# Check 1: Primary Key Uniqueness in Gold Table
gold_duplicates = df_gold.groupBy("customer_code").count().filter("count > 1").count()
print(f"Duplicate customer codes in Gold layer (Should be 0): {gold_duplicates}")

# Check 2: Total Records Count Comparison
silver_count = df_silver.count()
gold_count = df_gold.count()
print(f"Total customers in Silver: {silver_count}")
print(f"Total customers in Gold Dimension Table: {gold_count}")

# Preview Gold Layer Final Output
print("\n=== Gold Table Final Preview ===")
display(df_gold.limit(10))

=== LAYER 1: BRONZE (Source Audit) ===
Raw duplicate customer IDs in Bronze: 4
Raw records with untrimmed whitespace: 6

=== LAYER 2: SILVER (Cleansing Validation) ===
Remaining records with NULL city in Silver: 0
Distinct Cities in Silver Layer:
+---------+
|     city|
+---------+
|New Delhi|
|Hyderabad|
|Bengaluru|
+---------+


=== LAYER 3: GOLD (Dimensional Model Validation) ===
Duplicate customer codes in Gold layer (Should be 0): 0
Total customers in Silver: 35
Total customers in Gold Dimension Table: 53

=== Gold Table Final Preview ===


customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition
